# Vision Quest — Colab Runner (manual dataset upload)
Early Prediction of Batting Shot-Type from Cricket Videos (IE 643).

**FIRST: Runtime → Change runtime type → T4 GPU → Save.**
Then run the cells top to bottom. No Kaggle API needed — you upload the dataset zip yourself.

## Cell 1 — Confirm the GPU is on (should print a Tesla T4 line)

In [ ]:
!nvidia-smi

## Cell 2 — Clone your repo + install dependencies
Replace the URL with YOUR GitHub repo URL.

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USERNAME>/VisionQuest_IE643_CricketShotPrediction.git'
!git clone $REPO_URL
%cd VisionQuest_IE643_CricketShotPrediction
!pip install -q -r requirements.txt
print('\nSetup done.')

## Cell 3 — Upload the dataset zip
Run this cell, click **Choose Files**, pick the cricket-shots zip you downloaded.
Wait for it to finish uploading (shows 100%) before running the next cell.

In [ ]:
from google.colab import files
import os, glob
os.makedirs('data/raw', exist_ok=True)
up = files.upload()                      # pick your zip here
zipname = list(up.keys())[0]
print('uploaded:', zipname)
!unzip -q -o "$zipname" -d data/raw/
print('\nUnzipped. Top-level contents of data/raw:')
print(os.listdir('data/raw'))

## Cell 4 — Inspect the class folders  ⟵ STOP AND READ THE OUTPUT
This prints the shot-class folder names + clip counts. **Compare them to `classes:` in configs/default.yaml.**
If they differ, edit the config in Cell 5 before training. (Paste this output to your helper if unsure.)

In [ ]:
!python -m src.data.download --inspect --raw data/raw

## Cell 5 — (ONLY IF NEEDED) fix the class list / raw path
Skip this cell if Cell 4's class names already match the config.
Otherwise edit the list below to the EXACT folder names Cell 4 printed, and run it.

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/default.yaml'))
# --- EDIT this list to match Cell 4 exactly (lowercase, exact spelling) ---
cfg['classes'] = ['drive','pull','cut','sweep','flick','slog','misc']
# --- If Cell 4 said the class root was e.g. data/raw/CricketShots, set it here ---
cfg['paths']['raw_dataset'] = 'data/raw'
yaml.safe_dump(cfg, open('configs/default.yaml','w'), sort_keys=False)
print('Updated classes:', cfg['classes'])
print('raw_dataset:', cfg['paths']['raw_dataset'])

## Cell 6 — Preprocess: build split + cache frames, then extract pose
Takes a while (depends on dataset size). The frame step prints a progress bar.

In [ ]:
!python -m src.data.extract_frames --config configs/default.yaml --set paths.raw_dataset=data/raw
!python -m src.data.extract_pose   --config configs/default.yaml

## Cell 7 — Train the VIDEO branch (the main result)
If you're short on time, lower epochs/frames here. Watch the val top1 climb each epoch.

In [ ]:
!python -m src.train.train_video --config configs/default.yaml
# Faster run if needed:
# !python -m src.train.train_video --config configs/default.yaml --set train.epochs=15 data.num_frames=16

## Cell 8 — Train the POSE branch

In [ ]:
!python -m src.train.train_pose --config configs/default.yaml

## Cell 9 — Temporal analysis + figures (Subtask 4). This writes your results.

In [ ]:
!python -m src.eval.temporal_analysis --config configs/default.yaml
import json
r = json.load(open('results/temporal_results.json'))
print(json.dumps(r, indent=2)[:2000])

## Cell 10 — Show the headline figure

In [ ]:
from IPython.display import Image
Image('results/figures/accuracy_vs_ratio.png')

## Cell 11 — Download results + checkpoints to your computer
Grab this zip; it has the numbers (temporal_results.json), the figures, and the trained models.

In [ ]:
!zip -r vq_outputs.zip results checkpoints
from google.colab import files
files.download('vq_outputs.zip')

## Cell 12 — (For the DEMO video) launch the interface with a public link
Prints a public gradio.live URL. Open it, upload a clip, move the slider. Screen-record this for the demo video.

In [ ]:
!python -m app.gradio_app --config configs/default.yaml --share